Instructions: Using the F1 dataset, build a predictive model and log it in MLflow and write the ML model predictions into a database.

1. [20 pts] Create two (2) new tables in your own fatabse where you'll store the predictions from each model for this exercise.
2. [30 pts] Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifcats. Submit submit your MLflow experiments as part of your assignments
3. [30 pts] For each model, store its predictions in the corresponding table you created in your own database. Ensure you are using your own database to store your predictions.
4. [20 pts] Push your code to GitHub upon completion

In [0]:
# Homework 5, Question 2: Train & Log Two Models with MLflow (Random Forest & Logistic Regression)

# 1. Install MLflow 
%pip install mlflow

%pip install fsspec s3fs

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Obtaining dependency information for fsspec from https://files.pythonhosted.org/packages/44/4b/e0cfc1a6f17e990f3e64b7d941ddc4acdc7b19d6edd51abf495f32b1a9e4/fsspec-2025.3.2-py3-none-any.whl.metadata
  Obtaining dependency information for s3fs from https://files.pythonhosted.org/packages/66/e1/4db0388df5655de92ce5f9b60d2bef220a58dde130e0453e5433c579986e/s3fs-2025.3.2-py3-none-any.whl.metadata
  Obtaining dependency information for aiobotocore<3.0.0,>=2.5.4 from https://files.pythonhosted.org/packages/95/67/026598918f92145156f2feb7957f57daefda20375cc2ac1a0692a9b8010b/aiobotocore-2.21.1-py3-none-any.whl.metadata
  Obtaining dependency information for aiohttp!=4.0.0a0,!=4.0.0a1 from https://files.pythonhosted.org/packages/e2/ce/1a75384e01dd1bf546898b6062b1b5f7a59b6692ef802e4dd6db64fed264/aiohttp-3.11.18-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metada

In [0]:
# ------------ Homework 5 Q2: Train & Log Two Models with MLflow ------------

# 1. Imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

# 2. Load the F1 “results” dataset
df = pd.read_csv("s3://columbia-gr5069-main/raw/results.csv").dropna()


# 3. Create the binary target (finish_top3)
df["finish_top3"] = (df["positionOrder"] <= 3).astype(int)

# 4. Features and target
X = df[["grid", "points"]]
y = df["finish_top3"]

# 5. Train/Test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Set up your MLflow experiment
mlflow.set_experiment("/Users/sc5558@columbia.edu/f1_hw5_models")  # adjust your path

# 7. Define two models (random forest + logistic regression)+ hyperparameters
models = {
    "random_forest": {
        "estimator": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": 100,
            "max_depth": 10,
            "min_samples_split": 2,
            "min_samples_leaf": 1,
            "max_features": "sqrt"
        }
    },
    "logistic_regression": {
        "estimator": LogisticRegression(random_state=42, max_iter=1000),
        "params": {
            "C": 1.0,
            "penalty": "l2",
            "solver": "lbfgs"
        }
    }
}

# 8. Train, evaluate, and log each model
for name, info in models.items():
    model = info["estimator"].set_params(**info["params"])
    with mlflow.start_run(run_name=name):
        # Log hyperparameters
        mlflow.log_params(info["params"])
        
        # Train
        model.fit(X_train, y_train)
        
        # Predict & score
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred),
            "auc": roc_auc_score(y_test, y_proba)
        }
        mlflow.log_metrics(metrics)
        
        # Log the model artifact
        mlflow.sklearn.log_model(model, artifact_path="model")
        
        # Artifact 1: Confusion matrix plot
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(5,4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title(f"{name} Confusion Matrix")
        plt.xlabel("Predicted"); plt.ylabel("Actual")
        cm_path = f"{name}_confusion_matrix.png"
        plt.savefig(cm_path); plt.close()
        mlflow.log_artifact(cm_path)
        
        # Artifact 2: Feature importances (RF) or coefficient plot (LR)
        plt.figure(figsize=(6,4))
        if name == "random_forest":
            importances = model.feature_importances_
            idx = np.argsort(importances)
            plt.barh(range(len(importances)), importances[idx])
            plt.yticks(range(len(importances)), X.columns[idx])
            plt.title("Feature Importances")
            art_path = f"{name}_feature_importances.png"
        else:
            coefs = model.coef_[0]
            idx = np.argsort(np.abs(coefs))
            plt.barh(range(len(coefs)), coefs[idx])
            plt.yticks(range(len(coefs)), X.columns[idx])
            plt.title("Coefficient Values")
            art_path = f"{name}_coefficients.png"
        plt.tight_layout()
        plt.savefig(art_path); plt.close()
        mlflow.log_artifact(art_path)

print("✅ Done: both models trained and logged in MLflow.")


2025/04/30 01:31:54 INFO mlflow.tracking.fluent: Experiment with name '/Users/sc5558@columbia.edu/f1_hw5_models' does not exist. Creating a new experiment.
2025/04/30 01:32:01 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 2.22.0, required: mlflow==2.11.4)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
/databricks/python/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
2025/04/30 01:32:01 WARNING mlflow.models.model: Model logged without a signature. Signatures will be required for upcoming model registry features as they validate model inputs and denote the expected schema of model outputs. Please visit https://www.mlflow.org/docs/2.11

✅ Done: both models trained and logged in MLflow.


In [0]:
# --- Capture fitted models and build preds_1 & preds_2 ---

# 1) Retrain to get model objects
rf_model = None
lr_model = None

for name, info in models.items():
    model = info["estimator"].set_params(**info["params"])
    model.fit(X_train, y_train)
    if name == "random_forest":
        rf_model = model
    else:
        lr_model = model

# 2) Re-extract driverId, raceId, positionOrder for test set
X_test_meta = df.loc[X_test.index, ["driverId", "raceId", "positionOrder"]].reset_index(drop=True)

# 3) Build preds_1 for Random Forest
preds_1 = pd.DataFrame({
    "driverId":         X_test_meta.driverId,
    "raceId":           X_test_meta.raceId,
    "positionOrder":    X_test_meta.positionOrder,
    "predicted_top3":   rf_model.predict(X_test),
    "probability_top3": rf_model.predict_proba(X_test)[:, 1],
})
preds_1["actual_is_top3"] = (preds_1.positionOrder <= 3).astype(int)

# 4) Build preds_2 for Logistic Regression
preds_2 = pd.DataFrame({
    "driverId":         X_test_meta.driverId,
    "raceId":           X_test_meta.raceId,
    "positionOrder":    X_test_meta.positionOrder,
    "predicted_top3":   lr_model.predict(X_test),
    "probability_top3": lr_model.predict_proba(X_test)[:, 1],
})
preds_2["actual_is_top3"] = (preds_2.positionOrder <= 3).astype(int)

# 5) Quick sanity display
print("Random Forest predictions preview:")
display(preds_1.head())
print("Logistic Regression predictions preview:")
display(preds_2.head())

driverId,raceId,positionOrder,predicted_top3,probability_top3,actual_is_top3
501,799,6,0,0.0,0
55,182,14,0,0.0,0
404,730,9,0,0.0038763092093317218,0
333,607,8,0,0.0,0
140,310,29,0,0.0,0


driverId,raceId,positionOrder,predicted_top3,probability_top3,actual_is_top3
501,799,6,0,0.014689190038417474,0
55,182,14,0,0.017400939677493614,0
404,730,9,0,0.024379280382146002,0
333,607,8,0,0.0031557755234475006,0
140,310,29,0,0.09013450973838558,0


In [0]:
# ---------- Cell 6: Write Predictions to Aurora via mysql-connector-python ----------

# 1) Install the connector (if you haven’t already in this session)
%pip install mysql-connector-python

import mysql.connector

# 2) Open a connection to your Aurora MySQL
conn = mysql.connector.connect(
    host='sc5558-gr5069.ccqalx6jsr2n.us-east-1.rds.amazonaws.com',
    port=3306,
    user='admin',
    password='Angel20020419'  
)
cursor = conn.cursor()

# 3) Select the database
cursor.execute("USE gr5069")

# 4) Create the two prediction tables if they don’t exist already
cursor.execute("""
CREATE TABLE IF NOT EXISTS predictions_model_1 (
  driverId INT,
  raceId INT,
  positionOrder INT,
  predicted_top3 TINYINT,
  probability_top3 DOUBLE,
  actual_is_top3 TINYINT
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS predictions_model_2 (
  driverId INT,
  raceId INT,
  positionOrder INT,
  predicted_top3 TINYINT,
  probability_top3 DOUBLE,
  actual_is_top3 TINYINT
)
""")
conn.commit()

# 5) Build an INSERT statement and batch‐execute for each preds DataFrame
insert_sql = """
INSERT INTO {table}
  (driverId, raceId, positionOrder, predicted_top3, probability_top3, actual_is_top3)
VALUES (%s, %s, %s, %s, %s, %s)
"""

# preds_1 and preds_2 are the pandas DataFrames you created earlier
for table_name, df_preds in [
    ("predictions_model_1", preds_1),
    ("predictions_model_2", preds_2)
]:
    data = list(df_preds.itertuples(index=False, name=None))
    cursor.executemany(insert_sql.format(table=table_name), data)
    conn.commit()
    print(f"Inserted {cursor.rowcount} rows into {table_name}")

# 6) Clean up
cursor.close()
conn.close()

print("✅ All predictions written to Aurora MySQL!")
